In [1]:
import sys

print(sys.executable)

c:\Users\CES\Documents\DATA_ENGINEER\banking-dwh-pipeline\.venv\Scripts\python.exe


In [2]:
import pyspark

print("PySpark:", pyspark.__version__)


PySpark: 4.2.0


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("BankingDwhPipeline") \
    .config("spark.sql.shuffle.partitions", "4") \
    .master("local[*]") \
    .getOrCreate()

In [4]:
print("Aplicación:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Aplicación: BankingDwhPipeline
Master: local[*]
Shuffle partitions: 4


## Lectura del archivo `customers.csv`

En esta sección se crea un DataFrame a partir del archivo CSV de clientes.

Se utilizan las siguientes opciones:

- `header=True`: indica que la primera fila contiene los nombres de las columnas.
- `inferSchema=True`: permite que Spark intente detectar automáticamente los tipos de datos.
- `csv(...)`: indica que la fuente es un archivo CSV.

El DataFrame resultante se almacena en la variable `df_customers`.

In [5]:
df_customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/raw/customers.csv")
)

In [6]:
df_customers.head(5)

[Row(customer_id='CUSXAJI0Y6DPBHS', first_name='Kevin', last_name='Young', email='brauncameron@example.net', city='North Williamville', credit_score=327, created_at=datetime.date(2025, 4, 17)),
 Row(customer_id='CUSHXTHV3A3ZMF8', first_name='Jason', last_name='Clements', email='toddwilliam@example.net', city='Martinezside', credit_score=644, created_at=datetime.date(2020, 2, 23)),
 Row(customer_id='CUSDD4V30T9NT3W', first_name='Randy', last_name='Thompson', email='trevoranderson@example.org', city='Gallowayfurt', credit_score=670, created_at=datetime.date(2025, 6, 22)),
 Row(customer_id='CUSGCX1945NQ4FM', first_name='Laura', last_name='Phillips', email='valdezgeorge@example.com', city='Morrisview', credit_score=573, created_at=datetime.date(2019, 10, 20)),
 Row(customer_id='CUSVG0FN9XUY41I', first_name='Savannah', last_name='Swanson', email='smithluis@example.com', city='Lake Anna', credit_score=332, created_at=datetime.date(2022, 7, 16))]

In [7]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)



In [8]:
df_customers.select("*").show()

+---------------+----------+---------+--------------------+--------------------+------------+----------+
|    customer_id|first_name|last_name|               email|                city|credit_score|created_at|
+---------------+----------+---------+--------------------+--------------------+------------+----------+
|CUSXAJI0Y6DPBHS|     Kevin|    Young|brauncameron@exam...|  North Williamville|         327|2025-04-17|
|CUSHXTHV3A3ZMF8|     Jason| Clements|toddwilliam@examp...|        Martinezside|         644|2020-02-23|
|CUSDD4V30T9NT3W|     Randy| Thompson|trevoranderson@ex...|        Gallowayfurt|         670|2025-06-22|
|CUSGCX1945NQ4FM|     Laura| Phillips|valdezgeorge@exam...|          Morrisview|         573|2019-10-20|
|CUSVG0FN9XUY41I|  Savannah|  Swanson|smithluis@example...|           Lake Anna|         332|2022-07-16|
|CUSOC6UZHR5XFF0|    Ashley|     Wang|reesekendra@examp...|             Amybury|         569|2025-07-22|
|CUSPVN9ER14FFYV|    Morgan|   Miller| narnold@example.

## Inspección del esquema de `customers.csv`

El método `printSchema()` muestra la estructura del DataFrame, incluyendo:

- Nombre de cada columna.
- Tipo de dato detectado por Spark.
- Si la columna permite valores nulos.

El esquema obtenido fue:

```text
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)

In [9]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: date (nullable = true)




El siguiente paso lógico es conocer **cuántos clientes tiene el archivo**:

```python
total_customers = df_customers.count()

print("Total de clientes:", total_customers)

In [10]:
total_customers = df_customers.count()

print("Total de clientes:", total_customers)

Total de clientes: 50000


Ahora podemos aplicar lo visto en el curso con una primera selección:

```python
df_customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "credit_score"
).show(5, truncate=False)

In [11]:

df_customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "credit_score"
).show(5, truncate=False)

+---------------+----------+---------+------------+
|customer_id    |first_name|last_name|credit_score|
+---------------+----------+---------+------------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |327         |
|CUSHXTHV3A3ZMF8|Jason     |Clements |644         |
|CUSDD4V30T9NT3W|Randy     |Thompson |670         |
|CUSGCX1945NQ4FM|Laura     |Phillips |573         |
|CUSVG0FN9XUY41I|Savannah  |Swanson  |332         |
+---------------+----------+---------+------------+
only showing top 5 rows


## Ejercicio 1: selección de una sola columna

El método `select()` permite elegir una o varias columnas de un DataFrame.

En este ejercicio se seleccionará únicamente la columna `customer_id`.

```python
df_customers.select("customer_id").show(5, truncate=False)


In [12]:
df_clientes_id = df_customers.select(df_customers["customer_id"])
df_clientes_id.show(5, truncate=False)

+---------------+
|customer_id    |
+---------------+
|CUSXAJI0Y6DPBHS|
|CUSHXTHV3A3ZMF8|
|CUSDD4V30T9NT3W|
|CUSGCX1945NQ4FM|
|CUSVG0FN9XUY41I|
+---------------+
only showing top 5 rows


### Ejercicio 1: seleccionar el identificador del cliente

**Requerimiento:** seleccionar únicamente la columna `customer_id`.

El objetivo es practicar una proyección básica de una sola columna mediante el método `select()`.

Se deben mostrar los primeros cinco registros sin truncar el contenido.

In [13]:
df_customers.select("customer_id").show(5, truncate= False)

+---------------+
|customer_id    |
+---------------+
|CUSXAJI0Y6DPBHS|
|CUSHXTHV3A3ZMF8|
|CUSDD4V30T9NT3W|
|CUSGCX1945NQ4FM|
|CUSVG0FN9XUY41I|
+---------------+
only showing top 5 rows


### Ejercicio 2: seleccionar nombres y apellidos

**Requerimiento:** seleccionar las columnas `first_name` y `last_name`.

El ejercicio permite practicar la selección de varias columnas dentro de un mismo `select()`.

Las columnas deben aparecer en el mismo orden indicado en el requerimiento.

In [15]:
df_customers.select("first_name","last_name").show(5, truncate= False)

+----------+---------+
|first_name|last_name|
+----------+---------+
|Kevin     |Young    |
|Jason     |Clements |
|Randy     |Thompson |
|Laura     |Phillips |
|Savannah  |Swanson  |
+----------+---------+
only showing top 5 rows


### Ejercicio 3: seleccionar información de contacto

**Requerimiento:** seleccionar las columnas `customer_id`, `email` y `city`.

El objetivo es construir un nuevo DataFrame que contenga únicamente la identificación y la información básica de contacto de cada cliente.

Se deben mostrar los primeros cinco registros sin truncar los valores.

In [17]:
df_cliente_contacto = df_customers.select(
    "customer_id",
    "email",
    "city"
)

df_cliente_contacto.show(5, truncate= False)

+---------------+--------------------------+------------------+
|customer_id    |email                     |city              |
+---------------+--------------------------+------------------+
|CUSXAJI0Y6DPBHS|brauncameron@example.net  |North Williamville|
|CUSHXTHV3A3ZMF8|toddwilliam@example.net   |Martinezside      |
|CUSDD4V30T9NT3W|trevoranderson@example.org|Gallowayfurt      |
|CUSGCX1945NQ4FM|valdezgeorge@example.com  |Morrisview        |
|CUSVG0FN9XUY41I|smithluis@example.com     |Lake Anna         |
+---------------+--------------------------+------------------+
only showing top 5 rows


### Ejercicio 4: seleccionar todas las columnas

**Requerimiento:** seleccionar todas las columnas del DataFrame utilizando el carácter `*`.

Este ejercicio permite comprobar que `select("*")` proyecta la totalidad de las columnas disponibles.

El DataFrame original no debe ser modificado.

In [21]:
##Forma 1
df_customers_tot = df_customers.select(
    df_customers["*"]
)

df_customers_tot.show(n=5, truncate=False)

##Forma 2
df_customers_tot = df_customers.select("*")
df_customers_tot.show(n=5, truncate=False)

+---------------+----------+---------+--------------------------+------------------+------------+----------+
|customer_id    |first_name|last_name|email                     |city              |credit_score|created_at|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
|CUSXAJI0Y6DPBHS|Kevin     |Young    |brauncameron@example.net  |North Williamville|327         |2025-04-17|
|CUSHXTHV3A3ZMF8|Jason     |Clements |toddwilliam@example.net   |Martinezside      |644         |2020-02-23|
|CUSDD4V30T9NT3W|Randy     |Thompson |trevoranderson@example.org|Gallowayfurt      |670         |2025-06-22|
|CUSGCX1945NQ4FM|Laura     |Phillips |valdezgeorge@example.com  |Morrisview        |573         |2019-10-20|
|CUSVG0FN9XUY41I|Savannah  |Swanson  |smithluis@example.com     |Lake Anna         |332         |2022-07-16|
+---------------+----------+---------+--------------------------+------------------+------------+----------+
only showing top 5 

### Ejercicio 5: seleccionar una columna mediante un objeto `Column`

**Requerimiento:** seleccionar la columna `credit_score` utilizando la notación:

```python
df_customers["credit_score"]

In [22]:
df_customers_cs = df_customers.select(df_customers["credit_score"])
df_customers_cs.show(5, truncate=False)

+------------+
|credit_score|
+------------+
|327         |
|644         |
|670         |
|573         |
|332         |
+------------+
only showing top 5 rows


### Ejercicio 6: modificar el orden de las columnas

**Requerimiento:** seleccionar las columnas `last_name`, `first_name` y `customer_id`, en ese orden.

El ejercicio demuestra que el orden de las columnas en el resultado depende del orden en que se escriben dentro de `select()`.

No se modificará el orden ni la estructura del DataFrame original.

In [23]:
df_customers.select(
    "last_name",
    "first_name",
    "customer_id"
).show(5, truncate=False)

+---------+----------+---------------+
|last_name|first_name|customer_id    |
+---------+----------+---------------+
|Young    |Kevin     |CUSXAJI0Y6DPBHS|
|Clements |Jason     |CUSHXTHV3A3ZMF8|
|Thompson |Randy     |CUSDD4V30T9NT3W|
|Phillips |Laura     |CUSGCX1945NQ4FM|
|Swanson  |Savannah  |CUSVG0FN9XUY41I|
+---------+----------+---------------+
only showing top 5 rows


### Ejercicio 7: asignar un alias al identificador

**Requerimiento:** seleccionar la columna `customer_id` y asignarle temporalmente el alias `client_id`.

El objetivo es practicar el método `alias()` para cambiar el nombre mostrado de una columna dentro del DataFrame resultante.

El nombre original de la columna en `df_customers` no debe modificarse.

In [31]:
df_customers.select(
    df_customers["customer_id"].alias("client_id")
).show(5, truncate=False)




+---------------+
|client_id      |
+---------------+
|CUSXAJI0Y6DPBHS|
|CUSHXTHV3A3ZMF8|
|CUSDD4V30T9NT3W|
|CUSGCX1945NQ4FM|
|CUSVG0FN9XUY41I|
+---------------+
only showing top 5 rows


### Ejercicio 8: asignar un alias al puntaje crediticio

**Requerimiento:** seleccionar la columna `credit_score` y asignarle el alias `customer_credit_score`.

Este ejercicio permite practicar la creación de nombres más descriptivos para las columnas proyectadas.

El alias solo debe aplicarse al resultado del `select()`.

In [32]:
df_customers.select(
    df_customers["credit_score"].alias("customer_credit_score")).show(10, truncate=False)

+---------------------+
|customer_credit_score|
+---------------------+
|327                  |
|644                  |
|670                  |
|573                  |
|332                  |
|569                  |
|694                  |
|461                  |
|465                  |
|380                  |
+---------------------+
only showing top 10 rows


### Ejercicio 9: asignar un alias a la fecha de creación

**Requerimiento:** seleccionar la columna `created_at` y asignarle el alias `customer_creation_date`.

El objetivo es mejorar la claridad del nombre de la columna sin alterar el DataFrame original.

Se deben mostrar cinco registros.

In [34]:
df_customers.select(
    df_customers["created_at"]
   .alias("customer_creation_date")
    ).show(5, truncate=False)

+----------------------+
|customer_creation_date|
+----------------------+
|2025-04-17            |
|2020-02-23            |
|2025-06-22            |
|2019-10-20            |
|2022-07-16            |
+----------------------+
only showing top 5 rows


### Ejercicio 10: renombrar nombre y apellido

**Requerimiento:** seleccionar las columnas `first_name` y `last_name`.

Dentro del resultado:

- `first_name` debe mostrarse como `name`.
- `last_name` debe mostrarse como `surname`.

El ejercicio permite aplicar más de un alias dentro del mismo `select()`.

In [35]:
df_customers.select(
    df_customers["first_name"].alias("name"),
    df_customers["last_name"].alias("surname")
).show(5, truncate=False)

+--------+--------+
|name    |surname |
+--------+--------+
|Kevin   |Young   |
|Jason   |Clements|
|Randy   |Thompson|
|Laura   |Phillips|
|Savannah|Swanson |
+--------+--------+
only showing top 5 rows


### Ejercicio 11: aumentar el puntaje crediticio

**Requerimiento:** seleccionar:

- `customer_id`
- `credit_score`
- Una nueva expresión que sume 10 puntos a `credit_score`

La nueva columna debe llamarse `credit_score_plus_10`.

El objetivo es practicar operaciones aritméticas sobre objetos `Column` y asignar un alias al resultado.

In [38]:
df_customers.select(
    df_customers["customer_id"],
    df_customers["credit_score"],
    (df_customers["credit_score"]+10).alias("credit_score_plus_10")
).show(5, truncate=False)

+---------------+------------+--------------------+
|customer_id    |credit_score|credit_score_plus_10|
+---------------+------------+--------------------+
|CUSXAJI0Y6DPBHS|327         |337                 |
|CUSHXTHV3A3ZMF8|644         |654                 |
|CUSDD4V30T9NT3W|670         |680                 |
|CUSGCX1945NQ4FM|573         |583                 |
|CUSVG0FN9XUY41I|332         |342                 |
+---------------+------------+--------------------+
only showing top 5 rows


### Ejercicio 12: disminuir el puntaje crediticio

**Requerimiento:** seleccionar `customer_id` y calcular una nueva columna restando 50 puntos a `credit_score`.

La columna calculada debe llamarse:

```text
credit_score_minus_50

In [39]:
df_customers.select(
    df_customers["customer_id"],
    df_customers["credit_score"],
    (df_customers["credit_score"]-50).alias("credit_score_minus_50")
).show(5, truncate=False)

+---------------+------------+---------------------+
|customer_id    |credit_score|credit_score_minus_50|
+---------------+------------+---------------------+
|CUSXAJI0Y6DPBHS|327         |277                  |
|CUSHXTHV3A3ZMF8|644         |594                  |
|CUSDD4V30T9NT3W|670         |620                  |
|CUSGCX1945NQ4FM|573         |523                  |
|CUSVG0FN9XUY41I|332         |282                  |
+---------------+------------+---------------------+
only showing top 5 rows



### Ejercicio 13: duplicar el puntaje crediticio

**Requerimiento:** seleccionar `customer_id` y una expresión que multiplique `credit_score` por 2.

La nueva columna debe llamarse:

```text
double_credit_score

In [41]:
df_customers.select(df_customers["customer_id"],
                    df_customers["credit_score"],
                    (df_customers["credit_score"]*2).alias("double_credit_score")
                    ).show(5, truncate=False)

+---------------+------------+-------------------+
|customer_id    |credit_score|double_credit_score|
+---------------+------------+-------------------+
|CUSXAJI0Y6DPBHS|327         |654                |
|CUSHXTHV3A3ZMF8|644         |1288               |
|CUSDD4V30T9NT3W|670         |1340               |
|CUSGCX1945NQ4FM|573         |1146               |
|CUSVG0FN9XUY41I|332         |664                |
+---------------+------------+-------------------+
only showing top 5 rows


### Ejercicio 14: calcular una proporción del puntaje

**Requerimiento:** seleccionar `customer_id` y dividir `credit_score` entre 100.

El resultado debe almacenarse en una columna llamada:

```text
credit_score_ratio

In [43]:
df_customers.select(
    df_customers["customer_id"],
    df_customers["credit_score"],
    (df_customers["credit_score"]/100).alias("credit_score_ratio")
    ).show(5, truncate=False)

+---------------+------------+------------------+
|customer_id    |credit_score|credit_score_ratio|
+---------------+------------+------------------+
|CUSXAJI0Y6DPBHS|327         |3.27              |
|CUSHXTHV3A3ZMF8|644         |6.44              |
|CUSDD4V30T9NT3W|670         |6.7               |
|CUSGCX1945NQ4FM|573         |5.73              |
|CUSVG0FN9XUY41I|332         |3.32              |
+---------------+------------+------------------+
only showing top 5 rows


### Ejercicio 15: proyectar un puntaje futuro

**Requerimiento:** seleccionar:

- `first_name`
- `credit_score`
- Una expresión que sume 100 puntos a `credit_score`

La columna calculada debe llamarse:

```text
projected_credit_score

In [45]:
df_customers.select(
    df_customers["first_name"],
    df_customers["credit_score"],
    (df_customers["credit_score"]+100).alias("projected_credit_score")
    ).show(5, truncate=False)

+----------+------------+----------------------+
|first_name|credit_score|projected_credit_score|
+----------+------------+----------------------+
|Kevin     |327         |427                   |
|Jason     |644         |744                   |
|Randy     |670         |770                   |
|Laura     |573         |673                   |
|Savannah  |332         |432                   |
+----------+------------+----------------------+
only showing top 5 rows


### Ejercicio 16: convertir el nombre a mayúsculas

**Requerimiento:** seleccionar `first_name` y convertir su contenido completamente a mayúsculas.

La columna resultante debe llamarse:

```text
first_name_upper

In [48]:
from pyspark.sql import functions as F
df_customers.select(
    F.upper(df_customers["first_name"]).alias("first_name_upper")
    ).show(5, truncate=False)

+----------------+
|first_name_upper|
+----------------+
|KEVIN           |
|JASON           |
|RANDY           |
|LAURA           |
|SAVANNAH        |
+----------------+
only showing top 5 rows


### Ejercicio 17: convertir el apellido a minúsculas

**Requerimiento:** seleccionar `last_name` y convertir todos sus caracteres a minúsculas.

La columna resultante debe llamarse:

```text
last_name_lower

In [49]:
df_customers.select(
    F.lower(df_customers["last_name"]).alias("last_name_lower")
).show(5, truncate=False)

+---------------+
|last_name_lower|
+---------------+
|young          |
|clements       |
|thompson       |
|phillips       |
|swanson        |
+---------------+
only showing top 5 rows


### Ejercicio 18: calcular la longitud del nombre

**Requerimiento:** seleccionar `first_name` y calcular la cantidad de caracteres que contiene.

La columna calculada debe llamarse:

```text
first_name_length

In [51]:
df_customers.select(
    df_customers["first_name"],
    F.length(df_customers["first_name"]).alias("first_name_length")
).show(5, truncate=False)

+----------+-----------------+
|first_name|first_name_length|
+----------+-----------------+
|Kevin     |5                |
|Jason     |5                |
|Randy     |5                |
|Laura     |5                |
|Savannah  |8                |
+----------+-----------------+
only showing top 5 rows


### Ejercicio 19: construir el nombre completo

**Requerimiento:** crear una nueva columna uniendo `first_name` y `last_name`, separados por un espacio.

La nueva columna debe llamarse:

```text
full_name

In [59]:
df_customers.select(
    F.concat(df_customers["first_name"], F.lit(" "), df_customers["last_name"]).alias("full_name")
).show(5, truncate=False)

df_customers.select(
    F.concat_ws(" ", df_customers["first_name"], df_customers["last_name"]).alias("full_name")
).show(5, truncate=False)

+----------------+
|full_name       |
+----------------+
|Kevin Young     |
|Jason Clements  |
|Randy Thompson  |
|Laura Phillips  |
|Savannah Swanson|
+----------------+
only showing top 5 rows
+----------------+
|full_name       |
+----------------+
|Kevin Young     |
|Jason Clements  |
|Randy Thompson  |
|Laura Phillips  |
|Savannah Swanson|
+----------------+
only showing top 5 rows


### Ejercicio 20: construir una vista resumida del cliente

**Requerimiento:** crear un nuevo DataFrame con las siguientes columnas:

- `customer_id`
- Nombre completo del cliente
- `email`
- `city`
- `credit_score`

El nombre completo debe construirse uniendo `first_name` y `last_name` con un espacio y debe tener el alias:

```text
full_name

In [63]:
df_clients = df_customers.select(
    df_customers["customer_id"],
    F.concat_ws(" ", df_customers["first_name"], df_customers["last_name"]).alias("full_name"),
    "email",
    "city",
    "credit_score"
    )
df_clients.show(5, truncate=False)

+---------------+----------------+--------------------------+------------------+------------+
|customer_id    |full_name       |email                     |city              |credit_score|
+---------------+----------------+--------------------------+------------------+------------+
|CUSXAJI0Y6DPBHS|Kevin Young     |brauncameron@example.net  |North Williamville|327         |
|CUSHXTHV3A3ZMF8|Jason Clements  |toddwilliam@example.net   |Martinezside      |644         |
|CUSDD4V30T9NT3W|Randy Thompson  |trevoranderson@example.org|Gallowayfurt      |670         |
|CUSGCX1945NQ4FM|Laura Phillips  |valdezgeorge@example.com  |Morrisview        |573         |
|CUSVG0FN9XUY41I|Savannah Swanson|smithluis@example.com     |Lake Anna         |332         |
+---------------+----------------+--------------------------+------------------+------------+
only showing top 5 rows
